# Data Download Demo Walkthrough
---

-  This notebook provides a walkthrough of the data acquisition pipeline demonstrated in the Streamlit application. We'll go through each step of the process, explaining the functionality and showing code examples.

### Setup and Import Libraries

In [59]:
import os
import time
import json
import folium
import pandas as pd
import geopandas as gpd
from folium import Choropleth
from datetime import datetime
import matplotlib.pyplot as plt
from new_utils import gdf_from_geojson
from new_app import mask_downloaded_image, convert_mask_image_to_gdf
from IPython.display import display, Markdown
from data_downloader import (
    states_gdf_from_geojson, 
    get_available_dates, 
    dates_close_to_target_date, 
    get_dictionary_of_images_from_evalscripts, 
    get_total_polygon_from_gdf
)
from data_inference_collector import (
    get_square_list_for_state, 
    convert_square_to_polygon, 
    calculate_area_in_square_meters
)

def get_month_name(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d").strftime("%B")


def display_map(gdf, title, column=None, m=None, opacity=0.6):
    if len(gdf) > 1000:
        gdf = gdf.sample(1000)

    if m is None:
        centroid = gdf.geometry.centroid.iloc[0]
        m = folium.Map(location=[centroid.y, centroid.x], zoom_start=10)

    gdf.explore(column=column, style_kwds={"fillOpacity": opacity}, m=m)
    
    display(Markdown(f"### {title}"))
    display(m)

    return m



### Step 1: Select a State in Sudan

In [60]:
# First, we load the GeoJSON file containing Sudan states
print("# Step 1: Select a State in Sudan")
print("=" * 40)

sudan_gdf = states_gdf_from_geojson()
print(f"Loaded Sudan GeoDataFrame with states: {sudan_gdf['State'].tolist()}")

# Display the map of Sudan states
display_map(sudan_gdf, "Map of Sudan States", "State")

# In the application, the user would select a state. Here, we'll just select one
selected_state = "El Gazira"
print(f"Selected state: {selected_state}")


# Step 1: Select a State in Sudan
Loaded Sudan GeoDataFrame with states: ['Khartoum', 'River Nile', 'Northern', 'North Kordofan', 'West Kordofan', 'El Gazira', 'Sennar', 'White Nile', 'Kassala', 'Red Sea', 'Gedaref', 'South Kordofan', 'Abyei PCA Area', 'Blue Nile', 'Central Darfur', 'West Darfur', 'North Darfur', 'East Darfur', 'South Darfur']


/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### Map of Sudan States

Selected state: El Gazira


### Step 2: Show Selected State and Choose Square Size


In [61]:



print("\n# Step 2: Select Squares Size to cover the State")
print("=" * 50)

# Filter to show only the selected state
state_gdf = sudan_gdf[sudan_gdf["State"] == selected_state]
print(f"Filtered GeoDataFrame to show only {selected_state} state")

# Display the map of the selected state
display_map(state_gdf, f"Map of {selected_state} State")

# Choose square size parameters
max_height = 25  # km
max_width = 25   # km
print(f"Selected maximum square dimensions: {max_height}km × {max_width}km")
print("These parameters control the size of the grid squares used for analysis.")
print("Smaller squares provide more detail but create more computational load.")



# Step 2: Select Squares Size to cover the State
Filtered GeoDataFrame to show only El Gazira state


/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### Map of El Gazira State

Selected maximum square dimensions: 25km × 25km
These parameters control the size of the grid squares used for analysis.
Smaller squares provide more detail but create more computational load.


### Step 3: Generate and Display Squares


In [62]:


print("\n# Step 3: Generate Squares and Select a Specific One")
print("=" * 50)

# Generate squares
squares = get_square_list_for_state(state_gdf, max_height=max_height, max_width=max_width)
print(f"Generated {len(squares)} squares to cover {selected_state}")

# Convert squares to GeoDataFrame
geom = [convert_square_to_polygon(square) for square in squares]
squares_gdf = gpd.GeoDataFrame(geometry=geom)
squares_gdf.crs = "EPSG:4326"
squares_gdf['square_id'] = [f'{selected_state}_{i}' for i in range(len(squares_gdf))]
squares_gdf['Area_M2'] = squares_gdf['geometry'].apply(calculate_area_in_square_meters)
squares_gdf['Area_KM2'] = squares_gdf['Area_M2']/1000000
squares_gdf['location'] = selected_state
print(f"Created GeoDataFrame with {len(squares_gdf)} analysis squares")

# Display the squares
display_map(squares_gdf, f"{selected_state} with Analysis Squares")

# Select a specific square for analysis
square_index = 41
print(f"Selected square with index: {square_index}")


# Step 3: Generate Squares and Select a Specific One


100%|██████████| 15/15 [00:00<00:00, 97240.43it/s]

Generated 255 squares to cover El Gazira
Created GeoDataFrame with 255 analysis squares



/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### El Gazira with Analysis Squares

Selected square with index: 41


### Step 4: Select a Year and Find Available Dates

In [63]:
print("\n# Step 4: Select a Year and Find Available Dates")
print("=" * 50)

# Filter to the selected square
selected_square_gdf = squares_gdf[squares_gdf['square_id'] == f"{selected_state}_{square_index}"]
print(f"Selected square: {selected_state}_{square_index}")

# Display the selected square
display_map(selected_square_gdf, f"Selected Square: {selected_state}_{square_index}")

# Select a year for analysis
year = 2023
print(f"Selected year for analysis: {year}")

# Define target dates (one for each month in the rainy season)
pre_selected_dates = [
    f'{year}-06-01',
    f'{year}-07-16',
    f'{year}-08-05',
    f'{year}-09-19',
    f'{year}-10-29',
]
print(f"Pre-selected target dates: {pre_selected_dates}")

# Find available dates close to the target dates
available_dates = get_available_dates(selected_square_gdf, year)
print(f"Found {len(available_dates)} available dates in {year}")

# Get the closest available date to each target date
target_dates = []
for date in pre_selected_dates:
    closest_date = dates_close_to_target_date(dates=available_dates, target_date=date)[0]
    target_dates.append(closest_date)
    print(f"Found {closest_date} in {get_month_name(closest_date)} for year {year}")

print(f"Final selected dates: {target_dates}")


# Step 4: Select a Year and Find Available Dates
Selected square: El Gazira_41


/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### Selected Square: El Gazira_41

Selected year for analysis: 2023
Pre-selected target dates: ['2023-06-01', '2023-07-16', '2023-08-05', '2023-09-19', '2023-10-29']
dates fetched from cache
Found 73 available dates in 2023
Found 2023-06-01 in June for year 2023
Found 2023-07-16 in July for year 2023
Found 2023-08-05 in August for year 2023
Found 2023-09-19 in September for year 2023
Found 2023-10-29 in October for year 2023
Final selected dates: ['2023-06-01', '2023-07-16', '2023-08-05', '2023-09-19', '2023-10-29']


### Step 5: Choose Labels (GeoJSON)

In [64]:

print("\n# Step 5: Choose Labels GeoJSON")
print("=" * 50)

# Save the GeoJSON to a file (in the actual notebook, this would save to an actual file)
mask_path = "example.geojson"
print(f"Saved mask to {mask_path}")


# Step 5: Choose Labels GeoJSON
Saved mask to example.geojson


### Step 6: Show Labels with Selected Square

In [65]:


print("\n# Step 6: Show Labels with Selected Square")
print("=" * 50)

location_name = f'{selected_state}_{square_index}'
print(f"Location name: {location_name}")

# Load the mask GeoJSON
mask_gdf = gdf_from_geojson(geojson_path=mask_path, crs="EPSG:4326")
print(f"Loaded mask GeoDataFrame with {len(mask_gdf)} features")

# Display the map with the mask
display_map(mask_gdf, "Mask with Selected Square")

print("\nCurrent selections:")
print(f"State: {selected_state}")
print(f"Location Name: {location_name}")
print(f"Target Dates: {target_dates}")
display(f"Labels")
display(mask_gdf.to_wkt())



# Step 6: Show Labels with Selected Square
Location name: El Gazira_41
Loaded mask GeoDataFrame with 4 features


/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### Mask with Selected Square


Current selections:
State: El Gazira
Location Name: El Gazira_41
Target Dates: ['2023-06-01', '2023-07-16', '2023-08-05', '2023-09-19', '2023-10-29']


'Labels'

,label,geometry
0,Cultivated,"POLYGON ((32.707329 14.344309, 32.694025 14.34..."
1,Cultivated,"POLYGON ((32.721663 14.362104, 32.70484 14.316..."
2,Uncultivated,"POLYGON ((32.703724 14.369836, 32.692566 14.36..."
3,Uncultivated,"POLYGON ((32.736683 14.355867, 32.738914 14.36..."


### Step 7: Process and View Results

In [66]:
print("\n# Step 7: Process and View Results")
print("=" * 50)

# Process each date
evalscript = "ALL"  # Using all available evalscripts
print(f"Using evalscript: {evalscript}")

# Get the selected square
total_polygon = get_total_polygon_from_gdf(gdf=selected_square_gdf)
print(f"Got total polygon for processing")

# Process each date
month_geojson_dict = {}
for i, date in enumerate(target_dates):
    print(f"\nProcessing {get_month_name(date)} imagery ({i+1}/{len(target_dates)}):")
    
    # Download imagery
    print(f"Downloading satellite imagery...")
    download_dict = get_dictionary_of_images_from_evalscripts(
        total_polygon=total_polygon, 
        date=date, 
        location_name=location_name
    )
    
    # Apply mask
    print(f"Applying label mask to imagery...")
    mask_path = mask_downloaded_image(
        mask_gdf=mask_gdf, 
        location_name=location_name, 
        date=date, 
        evalscript=evalscript
    )
    
    # Convert mask to GeoDataFrame
    print(f"Converting results to GeoJSON...")
    geojson_path = convert_mask_image_to_gdf(
        location_name=location_name, 
        date=date, 
        evalscript=evalscript, 
        crs="EPSG:4326"
    )
    
    month = get_month_name(date)
    month_geojson_dict[month] = geojson_path
    print(f"Saved output for {month} to {geojson_path}")

print("\nDone! Aggregating results...")

# Aggregate results from all months
gdfs = []
for i, (month, geojson_path) in enumerate(month_geojson_dict.items()):
    gdf = gdf_from_geojson(geojson_path=geojson_path, crs="EPSG:4326")
    
    # Process column names
    for col in ['date', 'evalscript']:
        if col in gdf.columns:
            gdf = gdf.drop(columns=col)
    
    band_cols = [col for col in gdf.columns if 'band' in col]
    rename_dict = {col: f"{month}_{col}" for col in band_cols}
    gdf = gdf.rename(columns=rename_dict)
    
    if i > 0:
        if "location_name" in gdf.columns:
            gdf = gdf.drop(columns="location_name")
        if "geometry" in gdf.columns:
            gdf = gdf.drop(columns="geometry")
    
    gdfs.append(gdf)

# Combine all monthly data
result_gdf = pd.concat(gdfs, axis=1)
print(f"Final result table shape: {result_gdf.shape}")

# Display results
print("\nProcessed CSV (simplified for demonstration):")
print(result_gdf.head().to_string())

# Display on map
display_map(result_gdf, f"Processed Labels for {location_name}")

print("\n" + "=" * 50)
print("Analysis complete!")
print(f"Results show satellite data for {location_name} from months {', '.join(month_geojson_dict.keys())}")
print("The data can now be used for further analysis such as:")
print("=" * 50)


# Step 7: Process and View Results
Using evalscript: ALL
Got total polygon for processing

Processing June imagery (1/5):
Applying label mask to imagery...
Converting results to GeoJSON...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


Saved output for June to ./data/curated/El Gazira_41/ALL/2023-06-01/masked.geojson

Processing July imagery (2/5):
Applying label mask to imagery...
Converting results to GeoJSON...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


Saved output for July to ./data/curated/El Gazira_41/ALL/2023-07-16/masked.geojson

Processing August imagery (3/5):
Applying label mask to imagery...


...

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


Saved output for October to ./data/curated/El Gazira_41/ALL/2023-10-29/masked.geojson

Done! Aggregating results...
Final result table shape: (149078, 67)

Processed CSV (simplified for demonstration):
   June_band_1  June_band_2  June_band_3  June_band_4  June_band_5  June_band_6  June_band_7  June_band_8  June_band_9  June_band_10  June_band_11  June_band_12  June_band_13 location_name                   geometry  July_band_1  July_band_2  July_band_3  July_band_4  July_band_5  July_band_6  July_band_7  July_band_8  July_band_9  July_band_10  July_band_11  July_band_12  July_band_13  August_band_1  August_band_2  August_band_3  August_band_4  August_band_5  August_band_6  August_band_7  August_band_8  August_band_9  August_band_10  August_band_11  August_band_12  August_band_13  September_band_1  September_band_2  September_band_3  September_band_4  September_band_5  September_band_6  September_band_7  September_band_8  September_band_9  September_band_10  September_band_11  September

/var/folders/92/g_kswjtn2kd1qpqpdgl_k9xm0000gp/T/ipykernel_62710/1719906975.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = gdf.geometry.centroid.iloc[0]


### Processed Labels for El Gazira_41


Analysis complete!
Results show satellite data for El Gazira_41 from months June, July, August, September, October
The data can now be used for further analysis such as:


In [53]:
result_gdf.to_csv("demo_training_data.csv")